In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import normalize
from transformers import AutoTokenizer, AutoModel
import torch

# -------------------------------
# 基本配置
# -------------------------------
dataset_list = ['XTopic']
# dataset_list = ['20NG', 'AGNews', 'banking', 'clinc', 'DBPedia', 'ecdt', 'ele', 'hwu', 'mcid', 'news', 'stackoverflow', 'thucnews', 'TREC', 'X-Topic', 'Yahoo']

rates = [0.25, 0.5, 0.75]
fold_settings = [5]

EN_MODEL_PATH = '../pretrained_models/bert-base-uncased'
ZH_MODEL_PATH = '../pretrained_models/bert-base-chinese'
ZH_DATASETS = {'thucnews', 'ecdt'}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------------
# 函数定义
# -------------------------------
def get_model_path(dataset):
    return ZH_MODEL_PATH if dataset.lower() in ZH_DATASETS else EN_MODEL_PATH

@torch.no_grad()
def get_label_embeddings(labels, model_path):
    """用BERT编码标签文本并归一化"""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path).to(DEVICE)
    model.eval()
    vecs = []
    for i in range(0, len(labels), 64):
        batch = labels[i:i+64]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=16, return_tensors='pt').to(DEVICE)
        outputs = model(**inputs)
        attn = inputs['attention_mask'].unsqueeze(-1)
        emb = (outputs.last_hidden_state * attn).sum(dim=1) / attn.sum(dim=1)
        vecs.append(emb.cpu().numpy())
    E = np.vstack(vecs)
    return normalize(E, axis=1)

def farthest_first(E, k):
    """最远点采样选中心"""
    D = 1 - E @ E.T
    centers = [np.argmax(D.sum(1))]
    for _ in range(1, k):
        min_d = np.min(D[:, centers], axis=1)
        centers.append(np.argmax(min_d))
    return centers

# -------------------------------
# 主流程
# -------------------------------
for dataset in dataset_list:
    label_path = f'{dataset}/label/label.list'
    labels = pd.read_csv(label_path, header=None)[0].astype(str).tolist()
    N = len(labels)

    model_path = get_model_path(dataset)
    print(f'[{dataset}] Using model: {model_path}, total labels={N}')
    E = get_label_embeddings(labels, model_path)

    for fold_num in fold_settings:
        centers = farthest_first(E, fold_num)
        sim = E @ E[centers].T  # N×fold_num 余弦相似
        for rate in rates:
            # m = int(np.ceil(rate * N))
            m = int(max(round(N * rate), 1))
            for i, c in enumerate(centers):
                part_dir = Path(f'{dataset}/label/imbalance_fold{fold_num}/part{i}')
                part_dir.mkdir(parents=True, exist_ok=True)
                # 选取距离中心最近的 m 个标签（相似度最高）
                idx = np.argsort(-sim[:, i])[:m]
                chosen_labels = [labels[j] for j in idx]
                out_path = part_dir / f'label_known_{rate}.list'
                print('write to ', out_path)
                with open(out_path, 'w', encoding='utf-8') as f:
                    f.write('\n'.join(chosen_labels))
        print(f'  fold={fold_num} done.')

print("All done ✅")

/root/miniconda3/envs/bolt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[XTopic] Using model: ../pretrained_models/bert-base-uncased, total labels=64


2025-10-05 12:43:32.229115: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-05 12:43:32.285348: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-05 12:43:33.344549: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


write to  XTopic/label/imbalance_fold5/part0/label_known_0.25.list
write to  XTopic/label/imbalance_fold5/part1/label_known_0.25.list
write to  XTopic/label/imbalance_fold5/part2/label_known_0.25.list
write to  XTopic/label/imbalance_fold5/part3/label_known_0.25.list
write to  XTopic/label/imbalance_fold5/part4/label_known_0.25.list
write to  XTopic/label/imbalance_fold5/part0/label_known_0.5.list
write to  XTopic/label/imbalance_fold5/part1/label_known_0.5.list
write to  XTopic/label/imbalance_fold5/part2/label_known_0.5.list
write to  XTopic/label/imbalance_fold5/part3/label_known_0.5.list
write to  XTopic/label/imbalance_fold5/part4/label_known_0.5.list
write to  XTopic/label/imbalance_fold5/part0/label_known_0.75.list
write to  XTopic/label/imbalance_fold5/part1/label_known_0.75.list
write to  XTopic/label/imbalance_fold5/part2/label_known_0.75.list
write to  XTopic/label/imbalance_fold5/part3/label_known_0.75.list
write to  XTopic/label/imbalance_fold5/part4/label_known_0.75.list


In [2]:
import pandas as pd
import os
import json
import random
for dataset_name in dataset_list:
    for mode in ['train', 'dev']:
        df = pd.read_csv(f'{dataset_name}/origin_data/{mode}.tsv', sep='\t')
        for labeled_ratio in [0.1, 0.5, 1.0]:   
            df_group = df.groupby('label').agg(list)
            df_group['text'] = df_group['text'].apply(lambda x: random.sample(x, max(round(len(x) * labeled_ratio), 1)))
            df_group = df_group.reset_index().explode('text')[['text', 'label']]
            os.makedirs(f'{dataset_name}/labeled_data/{labeled_ratio}', exist_ok=True)
            known_text_list = df_group['text'].tolist()
            sub_df = df[:]
            sub_df['labeled'] = sub_df['text'].apply(lambda x: x in known_text_list)
            sub_df = sub_df.reset_index()[['label', 'labeled']]
            # sub_df.to_csv(f'{dataset_name}/labeled_data/{labeled_ratio}/{mode}.tsv', sep='\t', index=None)

/tmp/ipykernel_3327306/3764589135.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['labeled'] = sub_df['text'].apply(lambda x: x in known_text_list)
/tmp/ipykernel_3327306/3764589135.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['labeled'] = sub_df['text'].apply(lambda x: x in known_text_list)
/tmp/ipykernel_3327306/3764589135.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se